# Lab - Streaming: Show the Reply as It Is Written

Three tasks on delivering a reply word by word, as Claude writes it, instead of after the whole reply is done. A customer who waits at a blank screen thinks the chat is broken.

| Task | What you learn |
| --- | --- |
| 1 | Measure the wait: create() returns only when the whole reply is done |
| 2 | messages.stream(): print each piece as it arrives, then the final message |
| 3 | The same text, stop reason and usage - only the delivery changes |

**You do not need an API key.** This lab ships with a Claude simulator, so every cell runs offline. The setup cell makes simulated replies take time, like a real model writing token by token - otherwise there would be nothing to stream. The code you write is exactly the code you would write against the real API.

**You do not write code from scratch.** Each cell already holds the code, with blanks marked `...` and a comment telling you what goes in each one.

## Setup

Run this cell first, before anything else. Click it, then press **Shift+Enter**.

In [ ]:
# --- Lab setup (provided - just run it) ---
import shopassist_lab
from shopassist_lab import check

# Everything below this line is ordinary Claude API code.
from dotenv import load_dotenv
from anthropic import Anthropic
import time

load_dotenv()
client = Anthropic()
shopassist_lab.simulate_latency()   # offline only: replies take time, as a real model writes

# The model name lives in one constant, so a later update changes one line.
MODEL = "claude-sonnet-5-5"
SYSTEM = (
    "You are ShopAssist, an online store's support assistant. "
    "Returns are accepted within 30 days of delivery."
)
QUESTION = (
    "Explain in five short sentences how I return a jacket "
    "that arrived two weeks ago."
)
REQUEST = dict(
    model=MODEL,
    max_tokens=300,
    output_config={"effort": "low"},
    system=SYSTEM,
    messages=[{"role": "user", "content": QUESTION}],
)

---

## Task 1 - The wait without streaming

In [ ]:
# ============================================================
# TASK 1 - The wait without streaming
# ============================================================
#
# WHAT TO DO
#   Send the request the way you always have, with create(), and
#   time how long the script waits before it can print anything.
#
# WHY IT MATTERS
#   create() returns when the whole reply is done. For a short
#   answer that is a couple of seconds; for a long one, ten. A
#   customer who waits at a blank screen thinks the chat is
#   broken. Measure the wait first - it is the number streaming
#   is meant to fix.
#
# WHERE TO SEE IT IN THE LECTURE
#   "First, the way we have always done it".
#
# HOW TO DO IT
#   Three blanks:
#
#       REQUEST                   the request from the setup cell
#       t0                        the time before the call
#       response.content[0].text  the reply, printed all at once
# ============================================================

t0 = time.time()
try:
    # TODO: replace each ... below with the value named beside it
    response = client.messages.create(**...)   # REQUEST
    waited = time.time() - ...   # t0
    print(...)   # response.content[0].text
except (TypeError,) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
print(f"\n[whole reply after {waited:.1f} s - nothing to show until then]")

check("blocking_wait", waited=waited, response=response)

---

## Task 2 - Print the reply as it is written

In [ ]:
# ============================================================
# TASK 2 - Print the reply as it is written
# ============================================================
#
# WHAT TO DO
#   Send the same request with messages.stream(), in a with
#   block. Print each piece of text as it arrives, note when the
#   first one came, and take the final message at the end.
#
# WHY IT MATTERS
#   Claude writes one token after another either way. With
#   streaming, the API sends each piece as soon as it exists:
#   a message start, then block starts, many small deltas, block
#   stops, a message delta with the stop reason and usage, and a
#   message stop. The library turns those events into one loop,
#   text_stream - and get_final_message() hands you the same
#   object create() would have returned.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Instead of create, we call messages dot stream".
#
# HOW TO DO IT
#   Three blanks:
#
#       stream.text_stream           the pieces, as they arrive
#       None                         first is unset until the first piece
#       stream.get_final_message()   the complete message, after the loop
# ============================================================

pieces = []
first = None
t0 = time.time()
try:
    # TODO: replace each ... below with the value named beside it
    with client.messages.stream(**REQUEST) as stream:
        for text in ...:   # stream.text_stream
            if first is ...:   # None - only the first piece sets it
                first = time.time() - t0
            pieces.append(text)
            print(text, end="", flush=True)
        message = ...   # stream.get_final_message()
except (TypeError,) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
total = time.time() - t0
print(f"\n[first words after {first:.1f} s, whole reply after {total:.1f} s]")

check("stream_text", pieces=pieces, first=first, total=total, message=message)

---

## Task 3 - Only the delivery changes

In [ ]:
# ============================================================
# TASK 3 - Only the delivery changes
# ============================================================
#
# WHAT TO DO
#   Build the log record of the streamed call: the time to first
#   token, the total time, and stop_reason and output tokens from
#   the final message. Check that the pieces add up to the final
#   text.
#
# WHY IT MATTERS
#   It is tempting to think streaming changes what you get. It
#   does not: the final message has the same stop reason and the
#   same usage, and the text is the same text. Only the timing
#   changes - which is why a chat window streams and a batch job
#   that nobody watches does not. Time to first token is the
#   number a chat's users feel.
#
# WHERE TO SEE IT IN THE LECTURE
#   "It is tempting to think that streaming changes what we
#   get".
#
# HOW TO DO IT
#   Three blanks:
#
#       "".join(pieces)                the streamed text, put together
#       message.stop_reason            from the final message
#       message.usage.output_tokens    from the final message
# ============================================================

if first is None:   # provided: this task reads Task 2's results
    raise ValueError("Task 2 has not streamed a reply yet. Finish it and run it first.")
try:
    # TODO: replace each ... below with the value named beside it
    streamed = ...   # "".join(pieces)
    record = {
        "ttft": round(first, 2),
        "total": round(total, 2),
        "stop_reason": ...,     # message.stop_reason
        "output_tokens": ...,   # message.usage.output_tokens
    }
except (TypeError,) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
print("same text as the final message:", streamed == message.content[0].text)
print(record)

check("same_answer", pieces=pieces, message=message, record=record)

---

## Done

Stream when a person is waiting for the words: use messages.stream(), print each piece as it arrives, and read the stop reason and the usage from the final message, exactly as you would without streaming. Log the time to first token - it is what users feel.